# Maximal Rectangle

# Problem Statement

Given a binary matrix containing only `0` and `1`, find the area of the largest rectangle containing only `1`s.

### Input

A binary matrix:

```text
matrix
```

### Output

Return the area of the largest rectangle containing only `1`s.

### Example

```text
Input:

[
    ["1","0","1","0","0"],
    ["1","0","1","1","1"],
    ["1","1","1","1","1"],
    ["1","0","0","1","0"]
]

Output:

6
```

The largest rectangle of `1`s has:

```text
height = 2
width = 3
```

Therefore:

```text
Area = 2 × 3 = 6
```

# Problem Explanation

This is a 2D extension of:

```text
Largest Rectangle in Histogram
```

The key idea is to process the matrix **row by row**.

For every row, maintain an array:

```text
heights
```

where:

```text
heights[j]
```

represents the number of consecutive `1`s ending at the current row in column `j`.

Consider:

```text
1 0 1 1 1
1 1 1 1 1
```

After processing the first row:

```text
heights = [1,0,1,1,1]
```

After processing the second row:

```text
heights = [2,1,2,2,2]
```

Now this is simply a histogram.

So for every row:

```text
Build histogram
       ↓
Largest Rectangle in Histogram
       ↓
Update maximum area
```

This converts the 2D problem into repeated 1D problems.

# Brute Force

A brute-force solution can consider every possible rectangle.

For each possible pair of rows and columns, check whether all cells inside the rectangle are `1`.

This involves a large number of possible rectangles and can become extremely expensive.

A more direct improvement is to consider every cell as a possible top-left or bottom-right corner and expand the rectangle.

However, these approaches still require repeatedly checking matrix regions.

The important observation is that consecutive `1`s in each column can be represented as histogram heights.

### Why We Need Something Better

The histogram transformation allows us to reuse the linear-time:

```text
Largest Rectangle in Histogram
```

algorithm.

The resulting solution is much more efficient.

# Building the Histogram

Suppose the matrix is:

```text
1 0 1 1 1
1 1 1 1 1
1 1 1 1 0
```

Start:

```text
heights = [0,0,0,0,0]
```

### Row 1

```text
1 0 1 1 1
```

Update:

```text
heights = [1,0,1,1,1]
```

### Row 2

```text
1 1 1 1 1
```

Update:

```text
heights = [2,1,2,2,2]
```

### Row 3

```text
1 1 1 1 0
```

Update:

```text
heights = [3,2,3,3,0]
```

Every row therefore produces a histogram representing the consecutive `1`s ending at that row.

# Handling a Zero

If:

```text
matrix[i][j] == "1"
```

then:

```python
heights[j] += 1
```

because the consecutive sequence of `1`s continues.

If:

```text
matrix[i][j] == "0"
```

then:

```python
heights[j] = 0
```

because the consecutive sequence is broken.

This is the critical transformation:

```text
1 → increase height
0 → reset height
```

# Optimal Approach

For every row:

1. Update the histogram heights.
2. Find the largest rectangle in that histogram.
3. Update the global maximum.

The histogram calculation uses the same Monotonic Stack technique from:

```text
Largest Rectangle in Histogram
```

The Stack maintains increasing heights.

When a smaller height appears, taller bars are popped and their maximum possible rectangle areas are calculated.

Therefore:

```text
2D Matrix
    ↓
Histogram for each row
    ↓
Largest Rectangle in Histogram
    ↓
Maximum rectangle
```

In [1]:
class Solution:

    def largestRectangleArea(self, heights: list[int]) -> int:

        stack = []
        max_area = 0

        for i in range(len(heights) + 1):

            current_height = 0 if i == len(heights) else heights[i]

            while stack and current_height < heights[stack[-1]]:

                height = heights[stack.pop()]

                if stack:
                    left = stack[-1]
                else:
                    left = -1

                width = i - left - 1

                area = height * width

                max_area = max(max_area, area)

            stack.append(i)

        return max_area

In [2]:
class Solution:

    def maximalRectangle(self, matrix: list[list[str]]) -> int:

        if not matrix or not matrix[0]:
            return 0

        rows = len(matrix)
        cols = len(matrix[0])

        heights = [0] * cols
        max_area = 0

        for i in range(rows):

            for j in range(cols):

                if matrix[i][j] == "1":
                    heights[j] += 1
                else:
                    heights[j] = 0

            area = self.largestRectangleArea(heights)

            max_area = max(max_area, area)

        return max_area

# Dry Run

Consider:

```text
[
    [1,0,1,0,0],
    [1,0,1,1,1],
    [1,1,1,1,1],
    [1,0,0,1,0]
]
```

### Row 1

```text
1 0 1 0 0
```

Heights:

```text
[1,0,1,0,0]
```

Largest histogram rectangle:

```text
1
```

Maximum:

```text
1
```

---

### Row 2

```text
1 0 1 1 1
```

Heights:

```text
[2,0,2,1,1]
```

Largest rectangle:

```text
2 × 1 = 2
```

Maximum:

```text
2
```

---

### Row 3

```text
1 1 1 1 1
```

Heights:

```text
[3,1,3,2,2]
```

The rectangle:

```text
1 × 5 = 5
```

is possible.

Another rectangle:

```text
2 × 3 = 6
```

using columns `3,4,5`.

Therefore:

```text
Maximum = 6
```

---

### Row 4

```text
1 0 0 1 0
```

Heights:

```text
[4,0,0,3,0]
```

Largest rectangle:

```text
4
```

The global maximum remains:

```text
6
```

Final answer:

```text
6
```

# Why This Works

Every rectangle containing only `1`s has a bottom row.

Suppose its bottom row is:

```text
row i
```

For every column belonging to that rectangle, `heights[j]` tells us exactly how many consecutive `1`s extend upward from row `i`.

Therefore the rectangle becomes a rectangle in the histogram.

Since every possible rectangle has some bottom row, processing every row considers every possible rectangle.

That is why:

```text
Histogram of each row
```

is sufficient to solve the entire matrix problem.

# Edge Cases

### Empty Matrix

```text
Input:
[]

Output:
0
```

---

### Single Cell

```text
Input:
[["1"]]

Output:
1
```

---

### Single Zero

```text
Input:
[["0"]]

Output:
0
```

---

### All Zeros

```text
[
    ["0","0"],
    ["0","0"]
]
```

Output:

```text
0
```

---

### All Ones

```text
[
    ["1","1"],
    ["1","1"]
]
```

The entire matrix is one rectangle.

```text
Area = 2 × 2 = 4
```

# Common Mistakes

### Mistake 1 — Not Resetting a Column After `0`

When:

```text
matrix[i][j] == "0"
```

we must do:

```python
heights[j] = 0
```

Otherwise the histogram would incorrectly continue through the zero.

---

### Mistake 2 — Solving the Matrix Directly

Trying to repeatedly search the entire matrix makes the problem unnecessarily complicated.

Instead:

```text
Matrix
  ↓
Histogram
  ↓
Known Stack Problem
```

---

### Mistake 3 — Rebuilding the Histogram From Scratch

We do not need to recalculate all heights for every row.

Maintain:

```python
heights
```

and update each column incrementally.

---

### Mistake 4 — Forgetting the Global Maximum

Each row gives a different histogram.

The largest rectangle might occur at any row.

Therefore keep:

```python
max_area
```

across all rows.

# Complexity

Let:

```text
R = number of rows
C = number of columns
```

For every row:

```text
Update heights → O(C)
Largest Rectangle → O(C)
```

Therefore:

```text
O(C) + O(C) = O(C)
```

for each row.

Across all rows:

```text
Time → O(R × C)
```

The histogram and Stack require:

```text
Space → O(C)
```

So:

```text
Time  → O(RC)
Space → O(C)
```

# Comparison

| Approach | Time | Space |
|---|---:|---:|
| Brute Force | Very High | O(1) |
| Histogram + Stack | O(R × C) | O(C) |

The important improvement is that we reduce a difficult 2D rectangle problem into repeated linear-time histogram problems.

# Pattern Recognition

This problem has a very strong interview pattern:

```text
Binary Matrix
     +
Largest Rectangle
```

Think:

```text
Can I turn each row into histogram heights?
```

If yes:

```text
2D Matrix
    ↓
Histogram
    ↓
Largest Rectangle in Histogram
    ↓
Monotonic Stack
```

This is one of the most important examples of **reducing a complex problem to a problem you already know**.

# Connection With the Previous Problem

Previously:

```text
Largest Rectangle in Histogram
```

was a 1D problem.

Now:

```text
Maximal Rectangle
```

is a 2D problem.

But we do not need a completely new algorithm.

Instead:

```text
Maximal Rectangle
       ↓
Convert every row to histogram
       ↓
Reuse Largest Rectangle algorithm
```

This is an important problem-solving skill:

> Before inventing a new algorithm, check whether the problem can be transformed into a known problem.

# Takeaway

The core idea is:

```text
Each row becomes a histogram.
```

For every `1`:

```text
height += 1
```

For every `0`:

```text
height = 0
```

Then:

```text
largest histogram rectangle
```

is calculated using a Monotonic Stack.

The complete pattern is:

```text
Binary Matrix
      ↓
Running Column Heights
      ↓
Histogram
      ↓
Previous/Next Smaller
      ↓
Monotonic Stack
      ↓
Largest Rectangle
```

Complexity:

```text
Time  → O(R × C)
Space → O(C)
```

The major lesson is not just the Stack.

It is recognizing that a **2D problem can be transformed into repeated 1D problems**.